# MapQuery Annotation Exploration — JunB patient vs Healthy Controls

## Analysis Description

### Experimental design
Peripheral blood mononuclear cells (PBMCs) were profiled by single-cell RNA sequencing from
**7 healthy donors** (HD31, HD39, HD60, HD67, HD73, HD74, and one additional HD) and
**1 JunB-deficient patient (PID7)**, captured across two 10x Genomics chemistries:

| Chemistry | Samples | Donors |
|---|---|---|
| 3′ HT (v3.1) | sample01–03, 09–11, 16 | HD31, HD39, HD67, HD60, HD73, HD74, PID7 |
| 3′ GEM-X | sample17–20 | HD67, HD73, HD74, PID7 (technical replicates) |

Raw sequencing data were processed with **Cell Ranger** prior to this analysis.
All downstream steps were executed in a **Nextflow DSL2** pipeline with Docker containers.

---

### Step 1 — Per-sample quality control
Each of the 11 Cell Ranger outputs was loaded independently
(`filtered_feature_bc_matrix.h5` for HT; `sample_filtered_feature_bc_matrix.h5` for GEM-X)
using **Scanpy** (v1.10). Cells were retained if they expressed between **200 and 6,000 genes**
and had **< 15% mitochondrial reads**. Genes detected in fewer than **3 cells** were removed.
Doublets were identified and discarded with **Scrublet** (default parameters).

---

### Step 2 — Normalization and HVG selection (per sample)
Raw counts were stored in `adata.layers["counts"]` for downstream use.
Expression was normalised to **10,000 counts per cell** and log₁-transformed
(`sc.pp.normalize_total` + `sc.pp.log1p`).
**2,000 highly variable genes (HVGs)** were selected per sample using the Seurat v1 dispersion
method (`sc.pp.highly_variable_genes`, `flavor="seurat"`).
Scaling was intentionally deferred to the integration step to keep per-sample files sparse.

---

### Step 3 — Cross-sample merging
All 11 normalised AnnData objects were concatenated into a single object
(163,807 cells × 33,538 genes) with sample, patient ID, condition, and chemistry
metadata preserved per cell.

---

### Step 4 — Batch integration with Harmony
**2,000 HVGs** were re-selected on the merged object. The HVG subset was scaled
(clipped at 10 standard deviations) and **PCA** was computed (50 components).
**Harmony** batch correction (`harmonypy`, `vars_use=["sample"]`, max 20 iterations)
was applied to the PCA embedding to correct for sample- and chemistry-level effects,
producing a corrected `X_pca_harmony` representation.

---

### Step 5 — Clustering and UMAP
A **k-nearest-neighbour graph** was constructed on the Harmony embedding
(`sc.pp.neighbors`, default k = 15).
**UMAP** dimensionality reduction and **Leiden clustering** (resolution = 0.5) were performed,
yielding **18 clusters**.
Marker genes for each cluster were identified by Wilcoxon rank-sum test
(`sc.tl.rank_genes_groups`, top 20 genes per cluster).

---

### Step 6 — Cell type annotation

#### CellTypist (Immune_All_Low)
Automated annotation was performed with **CellTypist** using the
`Immune_All_Low.pkl` model (fine-grained immune atlas, majority voting enabled).
To handle the large cell count, 200 cells per Leiden cluster were subsampled for prediction;
labels were then propagated to all cells in each cluster by dominant-type assignment,
yielding **33 cell types**.

#### Seurat MapQuery (pbmc_multimodal)
The integrated object was converted to a Seurat object and annotated by
**label transfer** against the **PBMC Multimodal** reference (Hao et al. 2021, *Cell*)
using `FindTransferAnchors` + `TransferData` (SPCA-based, dims 1–50).
This produced **L1** (8 broad lineages) and **L2** (29 fine-grained subtypes) predictions
alongside per-cell mapping confidence scores.
Cells predicted as *Doublet* by MapQuery (n = 1) were excluded from downstream analyses.

---

### Step 7 — SCENIC gene regulatory network inference
**pySCENIC** (v0.12.1) was run on 5,000 randomly subsampled cells using 6,000 HVGs.

1. **GRN inference (GRNBoost2):** Co-expression networks were built for 1,892 human
   transcription factors; the top 300 targets per TF (by importance score) were retained.
2. **cisTarget motif pruning:** TF–target edges were filtered for cis-regulatory support
   using two ranking databases:
   - `hg38__refseq-r80__10kb_up_and_down_tss` (distal promoter/enhancer)
   - `hg38__refseq-r80__500bp_up_and_100bp_down_tss` (proximal promoter)
   The 80% gene-mapping threshold was applied per module.
3. **AUCell scoring:** Regulon activity (AUC) was scored across all 163,807 cells using
   only the regulon target genes to avoid memory constraints, yielding a
   **163,807 × 29 AUC matrix** (29 regulons).

AP1/JunB-related regulons recovered: **ATF3(+)**, **FOSL2(+)**, **MAFB(+)**, **NFE2(+)**.
JUNB itself did not pass the cisTarget 80% mapping threshold with the 6,000 HVG set;
its GRN co-expression adjacencies (3,249 edges) are retained for exploratory analysis.

---

### Software versions
Scanpy 1.10 · AnnData 0.10 · harmonypy · Scrublet · CellTypist · Seurat 5 ·
pySCENIC 0.12.1 · ctxcore 0.2 · Nextflow 24.10


In [ ]:
import pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

SCENIC_DIR = '../results_HVG6000/scenic'
SEURAT_DIR = '../results_HVG6000/annotate_seurat'
PALETTE    = {'HD': '#4878CF', 'PID7': '#D65F5F'}

## 1. Load SCENIC outputs

In [ ]:
adata = sc.read_h5ad(f'{SCENIC_DIR}/adata_scenic.h5ad')

with open(f'{SCENIC_DIR}/regulons.pkl', 'rb') as fh:
    regulons = pickle.load(fh)

auc_mtx = pd.read_csv(f'{SCENIC_DIR}/auc_matrix.csv', index_col=0)

regulon_names = list(adata.uns['scenic_regulon_names'])

# Align AUC to adata cell order and attach to obs
auc_aligned = auc_mtx.reindex(adata.obs_names)
for reg in regulon_names:
    adata.obs[reg] = auc_aligned[reg].values

AP1_RELATED = [r for r in regulon_names
               if any(k in r for k in ['ATF','JUN','FOS','BATF','MAF','NFE2'])]

print(f'Cells: {adata.n_obs:,}  |  Regulons: {len(regulons)}')
print(f'AP1/JunB regulons: {AP1_RELATED}')
print(f'Conditions: {adata.obs["condition"].value_counts().to_dict()}')

## 2. Load MapQuery labels

## 10. Seurat MapQuery annotation (pbmc_multimodal reference)

Labels transferred via Seurat `FindTransferAnchors` + `TransferData` against the
**PBMC Multimodal** reference (Hao et al. 2021, Cell).  
- **L1**: 8 broad lineages (B, CD4 T, CD8 T, DC, Mono, NK, other, other T)  
- **L2**: 29 fine-grained subtypes  
Compared below to CellTypist `Immune_All_Low` labels already in `adata.obs['cell_type']`.

In [ ]:
SEURAT_DIR = '../results_HVG6000/annotate_seurat'

mq = pd.read_csv(f'{SEURAT_DIR}/mapquery_labels.csv', index_col='barcode')

# Align to adata cell order
mq = mq.reindex(adata.obs_names)

adata.obs['mq_l1']       = mq['predicted.celltype.l1'].values
adata.obs['mq_l2']       = mq['predicted.celltype.l2'].values
adata.obs['mq_l2_score'] = mq['predicted.celltype.l2.score'].values

# Remove doublets from all downstream analyses
DOUBLET_LABELS = {'Doublet'}
doublet_mask = adata.obs['mq_l2'].isin(DOUBLET_LABELS)
n_doublets   = doublet_mask.sum()
adata.obs.loc[doublet_mask, ['mq_l1', 'mq_l2', 'mq_l2_score']] = np.nan
print(f'Removed {n_doublets:,} doublet cells from MapQuery labels (set to NaN)')

print(f'MapQuery labels: {adata.obs["mq_l2"].notna().sum():,} / {adata.n_obs:,} cells')
print(f'\nL1 counts:\n{adata.obs["mq_l1"].value_counts().to_string()}')
print(f'\nL2 unique labels: {adata.obs["mq_l2"].nunique()}')

### Leiden cluster numbers

In [ ]:
# ── Leiden cluster UMAP ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
sc.pl.umap(adata, color='leiden', ax=ax, show=False, frameon=False,
           legend_loc='none', title='Leiden clusters', size=0.5,
           palette='tab20')

umap   = adata.obsm['X_umap']
groups = adata.obs['leiden'].astype(str)

for cluster in sorted(groups.unique(), key=int):
    mask = groups == cluster
    cx, cy = umap[mask, 0].mean(), umap[mask, 1].mean()
    ax.text(cx, cy, cluster, fontsize=9, ha='center', va='center',
            fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.15', fc='white',
                      ec='#888888', alpha=0.75, linewidth=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# ── Labeled UMAP panels ───────────────────────────────────────────────────────
import matplotlib.patheffects as pe
from adjustText import adjust_text

def labeled_umap(adata, color_key, ax, title, palette=None,
                 fontsize=8, dot_size=1, outline=True):
    """Plot UMAP with cell type labels at cluster centroids, repelled by adjustText."""
    import numpy as np

    sc.pl.umap(adata, color=color_key, ax=ax, show=False, frameon=False,
               title=title, legend_loc='none', size=dot_size,
               palette=palette)

    umap   = adata.obsm['X_umap']
    groups = adata.obs[color_key].astype(str)

    texts = []
    for grp in groups.unique():
        if grp in ('nan', 'NA'):
            continue
        mask = groups == grp
        cx, cy = umap[mask, 0].mean(), umap[mask, 1].mean()
        txt = ax.text(cx, cy, grp, fontsize=fontsize,
                      ha='center', va='center', fontweight='bold',
                      color='black')
        if outline:
            txt.set_path_effects([
                pe.withStroke(linewidth=2.5, foreground='white')
            ])
        texts.append(txt)

    adjust_text(
        texts, ax=ax,
        arrowprops=dict(arrowstyle='-', color='#555555', lw=0.5),
        expand=(1.15, 1.4),
        force_text=(0.3, 0.5),
        force_points=(0.1, 0.2),
        lim=300,
    )
    return ax


# ── 1. CellTypist (Immune_All_Low) ───────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(11, 9))
labeled_umap(adata, 'cell_type', ax1,
             title='CellTypist  (Immune_All_Low)',
             fontsize=7, dot_size=0.5)
plt.tight_layout()
plt.show()

# ── 2. MapQuery L1 ───────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(9, 7))
labeled_umap(adata, 'mq_l1', ax2,
             title='MapQuery L1  (pbmc_multimodal)',
             fontsize=9, dot_size=0.5)
plt.tight_layout()
plt.show()

# ── 3. MapQuery L2 ───────────────────────────────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(12, 10))
labeled_umap(adata, 'mq_l2', ax3,
             title='MapQuery L2  (pbmc_multimodal)',
             fontsize=7.5, dot_size=0.5)
plt.tight_layout()
plt.show()

# ── 4. Mapping score ─────────────────────────────────────────────────────────
fig4, ax4 = plt.subplots(figsize=(7, 6))
sc.pl.umap(adata, color='mq_l2_score', ax=ax4,
           title='MapQuery L2 mapping score',
           color_map='viridis', show=False, frameon=False,
           colorbar_loc='right', size=0.5)
plt.tight_layout()
plt.show()

### HD vs PID7 — split UMAP

In [ ]:
# ── HD / PID7 split UMAP — 20,000 cells per condition ───────────────────────
# Subsample each condition independently to 20k cells so both panels are
# equally populated and dot density is comparable.

N_SAMPLE    = 20_000
SEED        = 42
conditions  = ['HD', 'PID7']
titles_l1   = ['MapQuery L1  —  HD', 'MapQuery L1  —  PID7 (JunB)']
titles_l2   = ['MapQuery L2  —  HD', 'MapQuery L2  —  PID7 (JunB)']
titles_ct   = ['CellTypist  —  HD',  'CellTypist  —  PID7 (JunB)']

rng = np.random.default_rng(SEED)

def subsample_condition(adata, cond, n):
    """Return up to n randomly sampled cells from condition cond."""
    idx = np.where(adata.obs['condition'] == cond)[0]
    if len(idx) > n:
        idx = rng.choice(idx, n, replace=False)
    return adata[idx]

# Build per-condition subsamples once; reuse across all plot calls
subs = {cond: subsample_condition(adata, cond, N_SAMPLE) for cond in conditions}
for cond, sub in subs.items():
    print(f'{cond}: {sub.n_obs:,} cells sampled')

# Shared colour maps so every cluster keeps the same colour across panels
palette_l1 = dict(zip(
    sorted(adata.obs['mq_l1'].dropna().unique()),
    plt.cm.tab10.colors[:adata.obs['mq_l1'].nunique()]
))
palette_l2 = {ct: col for ct, col in
              zip(sorted(adata.obs['mq_l2'].dropna().unique()),
                  plt.cm.tab20.colors + plt.cm.tab20b.colors)}

def split_labeled_umap(subs, color_key, titles, palette,
                       fontsize=7, dot_size=2, figsize=(20, 9)):
    """Two-panel labeled UMAP split by condition from pre-subsampled adatas."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, cond, title in zip(axes, conditions, titles):
        sub = subs[cond]
        sc.pl.umap(sub, color=color_key, ax=ax, show=False, frameon=False,
                   title=title, legend_loc='none', size=dot_size,
                   palette=palette, na_color='#dddddd')
        umap   = sub.obsm['X_umap']
        groups = sub.obs[color_key].astype(str)
        texts  = []
        for grp in groups.dropna().unique():
            if grp in ('nan', 'NA'):
                continue
            mask = groups == grp
            if mask.sum() < 5:
                continue
            cx, cy = umap[mask, 0].mean(), umap[mask, 1].mean()
            txt = ax.text(cx, cy, grp, fontsize=fontsize,
                          ha='center', va='center', fontweight='bold')
            txt.set_path_effects([pe.withStroke(linewidth=2.5, foreground='white')])
            texts.append(txt)
        adjust_text(texts, ax=ax,
                    arrowprops=dict(arrowstyle='-', color='#777', lw=0.4),
                    expand=(1.1, 1.3), force_text=(0.25, 0.4), lim=200)
        ax.text(0.02, 0.98, f'n = {sub.n_obs:,}',
                transform=ax.transAxes, fontsize=8,
                va='top', ha='left', color='#333333')
    plt.tight_layout()
    plt.show()


# ── L1 split ─────────────────────────────────────────────────────────────────
split_labeled_umap(subs, 'mq_l1', titles_l1, palette_l1, fontsize=9)

# ── L2 split ─────────────────────────────────────────────────────────────────
split_labeled_umap(subs, 'mq_l2', titles_l2, palette_l2, fontsize=7)

# ── CellTypist split ──────────────────────────────────────────────────────────
split_labeled_umap(subs, 'cell_type', titles_ct, None, fontsize=6.5)

### 10a. CellTypist vs MapQuery label concordance

In [ ]:
# Confusion matrix: CellTypist (rows) vs MapQuery L2 (cols)
conf = pd.crosstab(
    adata.obs['cell_type'].astype(str),
    adata.obs['mq_l2'].astype(str),
    normalize='index',
)

fig, ax = plt.subplots(figsize=(max(14, conf.shape[1] * 0.55),
                                max(8,  conf.shape[0] * 0.4)))
sns.heatmap(conf, cmap='Blues', ax=ax, linewidths=0.2, linecolor='lightgrey',
            cbar_kws={'label': 'Fraction of CellTypist cells'},
            xticklabels=True, yticklabels=True)
ax.set_xlabel('MapQuery L2')
ax.set_ylabel('CellTypist')
ax.set_title('CellTypist vs MapQuery L2 — row-normalised')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

### 10b. MapQuery L2 regulon AUC activity

In [ ]:
# Mean AUC per MapQuery L2 cell type — clustered heatmap
mean_auc_mq = (
    adata.obs[['mq_l2'] + regulon_names]
    .dropna(subset=['mq_l2'])
    .groupby('mq_l2')[regulon_names]
    .mean()
)
mean_auc_mq_z = mean_auc_mq.apply(lambda x: (x - x.mean()) / (x.std() + 1e-8))

g = sns.clustermap(
    mean_auc_mq_z,
    cmap='RdBu_r', center=0,
    figsize=(max(10, len(regulon_names) * 0.6), max(8, len(mean_auc_mq) * 0.4)),
    linewidths=0.2, dendrogram_ratio=0.1,
    cbar_pos=(1.01, 0.3, 0.03, 0.4),
    xticklabels=True, yticklabels=True,
)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=8)
for tick in g.ax_heatmap.get_xticklabels():
    if tick.get_text() in AP1_RELATED:
        tick.set_color('#D65F5F')
        tick.set_fontweight('bold')
g.fig.suptitle('Mean AUC per MapQuery L2 cell type (z-scored, red = AP1/JunB)', y=1.01)
plt.show()

### 10c. AP1/JunB regulons — PID7 vs HD by MapQuery L2 cell type

In [ ]:
obs_mq = adata.obs[['mq_l2', 'condition'] + AP1_RELATED].dropna(subset=['mq_l2'])

fig, axes = plt.subplots(1, len(AP1_RELATED),
                         figsize=(len(AP1_RELATED) * 5,
                                  max(5, obs_mq['mq_l2'].nunique() * 0.35)))
if len(AP1_RELATED) == 1:
    axes = [axes]

for ax, reg in zip(axes, AP1_RELATED):
    ct_cond = (
        obs_mq.groupby(['mq_l2', 'condition'])[reg]
        .mean()
        .unstack('condition')
        .assign(diff=lambda d: d.get('PID7', 0) - d.get('HD', 0))
        .sort_values('diff', ascending=False)
    )
    colors = ['#D65F5F' if v > 0 else '#4878CF' for v in ct_cond['diff']]
    ax.barh(ct_cond.index, ct_cond['diff'], color=colors)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Mean AUC (PID7 − HD)')
    ax.set_title(reg, color='#D65F5F')
    ax.tick_params(axis='y', labelsize=7)

plt.suptitle('AP1/JunB regulon: PID7 vs HD — MapQuery L2 cell types', y=1.01)
plt.tight_layout()
plt.show()

### Regulon activity violin plots — by cell type and condition

Three views:
1. **AP1/JunB regulons** — distribution across the top 8 cell types by RSS, split HD vs PID7
2. **Myeloid focus** — all AP1/JunB regulons in monocytes/macrophages/DC, split HD vs PID7
3. **All regulons overview** — box + strip across all MapQuery L2 types for each regulon

In [ ]:
rss_mq = pd.read_csv(f'{SCENIC_DIR}/rss_scores.csv', index_col=0).fillna(0)

obs_full = adata.obs[['mq_l2', 'condition'] + regulon_names].dropna(subset=['mq_l2']).copy()
obs_full['mq_l2'] = obs_full['mq_l2'].astype(str)

MYELOID = [c for c in obs_full['mq_l2'].unique()
           if any(k in c.lower() for k in ['mono', 'macro', 'dc', 'dendrit'])]

# ── 1. AP1/JunB regulons — top 8 cell types per regulon ──────────────────────
for reg in AP1_RELATED:
    # pick top 8 cell types by RSS for this regulon
    if reg in rss_mq.columns:
        top_cts = rss_mq[reg].nlargest(8).index.tolist()
        top_cts = [ct for ct in top_cts if ct in obs_full['mq_l2'].unique()]
    else:
        top_cts = obs_full['mq_l2'].value_counts().head(8).index.tolist()

    df = obs_full[obs_full['mq_l2'].isin(top_cts)][['mq_l2', 'condition', reg]].copy()

    fig, ax = plt.subplots(figsize=(max(10, len(top_cts) * 1.4), 4))
    sns.violinplot(
        data=df, x='mq_l2', y=reg, hue='condition',
        hue_order=['HD', 'PID7'], palette=PALETTE,
        split=True, inner='quartile', cut=0,
        order=top_cts, ax=ax, linewidth=0.8,
    )
    ax.set_title(f'{reg} — top 8 cell types by RSS (split: HD | PID7)',
                 color='#D65F5F')
    ax.set_xlabel('')
    ax.set_ylabel('AUC score')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.legend(title='Condition', fontsize=8, title_fontsize=8)
    plt.tight_layout()
    plt.show()

# ── 2. Myeloid focus — all AP1/JunB regulons ─────────────────────────────────
df_mono = obs_full[obs_full['mq_l2'].isin(MYELOID)].copy()
# melt to long form so we can facet by regulon
df_long = df_mono.melt(
    id_vars=['mq_l2', 'condition'],
    value_vars=AP1_RELATED,
    var_name='regulon', value_name='AUC',
)

fig, axes = plt.subplots(1, len(AP1_RELATED),
                         figsize=(len(AP1_RELATED) * 4, max(4, len(MYELOID) * 0.55)))
for ax, reg in zip(axes, AP1_RELATED):
    df_r = df_long[df_long['regulon'] == reg]
    sns.violinplot(
        data=df_r, y='mq_l2', x='AUC', hue='condition',
        hue_order=['HD', 'PID7'], palette=PALETTE,
        split=True, inner='quartile', cut=0,
        order=sorted(MYELOID), ax=ax, linewidth=0.8, orient='h',
    )
    ax.set_title(reg, color='#D65F5F', fontsize=9)
    ax.set_ylabel('')
    ax.set_xlabel('AUC', fontsize=8)
    ax.tick_params(axis='y', labelsize=7)
    ax.get_legend().remove() if ax != axes[-1] else ax.legend(
        title='Condition', fontsize=7, title_fontsize=7)

plt.suptitle('AP1/JunB regulons — myeloid cells, HD vs PID7', y=1.02)
plt.tight_layout()
plt.show()

# ── 3. All regulons overview — violin per regulon, colour = condition ─────────
# Use only active regulons (non-zero AUC)
active_regs = [r for r in regulon_names if obs_full[r].max() > 1e-5]
ncols = 5
nrows = int(np.ceil(len(active_regs) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3))
axes = axes.flatten()

for i, reg in enumerate(active_regs):
    df_r = obs_full[['mq_l2', 'condition', reg]].copy()
    # order cell types by mean AUC descending
    order = (df_r.groupby('mq_l2')[reg].mean()
                 .sort_values(ascending=False).index.tolist())
    sns.violinplot(
        data=df_r, x='mq_l2', y=reg, hue='condition',
        hue_order=['HD', 'PID7'], palette=PALETTE,
        split=True, inner='box', cut=0,
        order=order, ax=axes[i], linewidth=0.6,
    )
    title_color = '#D65F5F' if reg in AP1_RELATED else 'black'
    axes[i].set_title(reg + (' ★' if reg in AP1_RELATED else ''),
                      fontsize=8, color=title_color)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('AUC', fontsize=7)
    axes[i].tick_params(axis='x', rotation=90, labelsize=5)
    axes[i].get_legend().remove()

# shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', title='Condition',
           fontsize=8, title_fontsize=8)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('All active regulons — AUC by MapQuery L2 cell type (★ = AP1/JunB)',
             y=1.01, fontsize=10)
plt.tight_layout()
plt.show()

---
## CD4 T cell subset focus

Five MapQuery L2 subtypes: **CD4 Naive**, **CD4 TCM**, **CD4 TEM**,
**CD4 Proliferating**, **Treg**.  
Key regulons in this compartment: LEF1(+), BCL6(+), FOXO3(+), ATF3(+),
TBX21(+), RUNX3(+), and the AP1-related ATF3/FOSL2/MAFB.

In [ ]:
CD4_TYPES = ['CD4 Naive', 'CD4 Proliferating', 'CD4 TCM', 'CD4 TEM', 'Treg']
CD4_ORDER = ['CD4 Naive', 'CD4 TCM', 'CD4 TEM', 'CD4 Proliferating', 'Treg']

# Subset adata to CD4 cells
cd4_mask  = adata.obs['mq_l2'].isin(CD4_TYPES)
adata_cd4 = adata[cd4_mask].copy()

print('CD4 T cell subset counts:')
ct = (adata_cd4.obs.groupby(['mq_l2', 'condition'])
      .size().unstack(fill_value=0))
print(ct.to_string())
print(f'\nTotal CD4 cells: {adata_cd4.n_obs:,}')

### UMAP — CD4 subsets highlighted

In [ ]:
# Grey-out non-CD4 cells; colour CD4 subtypes
import matplotlib.patches as mpatches

cd4_palette = dict(zip(CD4_ORDER, plt.cm.Set2.colors[:len(CD4_ORDER)]))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, title, mask in [
    (axes[0], 'CD4 T cell subsets (all cells)', slice(None)),
    (axes[1], 'CD4 T cell subsets — split view', slice(None)),
]:
    # All cells in grey
    sc.pl.umap(adata, ax=axes[0], show=False, frameon=False,
               color='mq_l2', legend_loc='none', size=0.3,
               na_color='#e0e0e0', title='CD4 subsets highlighted',
               palette={ct: ('#e0e0e0' if ct not in CD4_TYPES
                             else cd4_palette[ct])
                        for ct in adata.obs['mq_l2'].dropna().unique()})
    break  # one panel approach

# Overlay CD4 cells on top with larger dots
umap_coords = adata.obsm['X_umap']
for ct in CD4_ORDER:
    idx = adata.obs['mq_l2'] == ct
    axes[0].scatter(umap_coords[idx, 0], umap_coords[idx, 1],
                    c=[cd4_palette[ct]], s=2, label=ct, zorder=3)
axes[0].legend(handles=[mpatches.Patch(color=cd4_palette[ct], label=ct)
                         for ct in CD4_ORDER],
               fontsize=7, loc='lower right', title='CD4 subtype')

# Right panel: CD4 cells only, coloured
sc.pl.umap(adata_cd4, color='mq_l2', ax=axes[1], show=False, frameon=False,
           title='CD4 cells only', legend_loc='on data',
           legend_fontsize=8, size=2,
           palette=cd4_palette)

plt.tight_layout()
plt.show()

### Cell count per subtype and condition

In [ ]:
ct_df = (adata_cd4.obs.groupby(['mq_l2', 'condition'])
         .size().reset_index(name='n'))
ct_df['mq_l2'] = pd.Categorical(ct_df['mq_l2'], categories=CD4_ORDER, ordered=True)
ct_df = ct_df.sort_values('mq_l2')

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(CD4_ORDER))
w = 0.35
for i, cond in enumerate(['HD', 'PID7']):
    vals = [ct_df.query('mq_l2 == @s and condition == @cond')['n'].sum()
            for s in CD4_ORDER]
    ax.bar(x + (i - 0.5) * w, vals, w, label=cond,
           color=PALETTE[cond], alpha=0.85)
    for xi, v in zip(x + (i - 0.5) * w, vals):
        if v > 0:
            ax.text(xi, v + 20, str(v), ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(CD4_ORDER, rotation=20, ha='right')
ax.set_ylabel('Cell count')
ax.set_title('CD4 T cell subset counts — HD vs PID7')
ax.legend(title='Condition')
plt.tight_layout()
plt.show()

### Marker gene expression across CD4 subtypes

In [ ]:
CD4_MARKERS = {
    'Identity': ['CCR7', 'LEF1', 'GZMK', 'GZMB', 'FOXP3', 'IL2RA'],
    'Activation / Cytokine': ['IFNG', 'TNF', 'BCL6', 'CXCR5'],
    'AP1/JunB':  ['JUNB', 'JUN', 'FOS', 'ATF3', 'FOSL2', 'MAFB'],
}
all_markers = [g for group in CD4_MARKERS.values() for g in group
               if g in adata_cd4.var_names]

sc.pl.dotplot(
    adata_cd4,
    var_names={k: [g for g in v if g in adata_cd4.var_names]
               for k, v in CD4_MARKERS.items()},
    groupby='mq_l2',
    categories_order=CD4_ORDER,
    standard_scale='var',
    title='CD4 T cell subset marker genes',
    figsize=(13, 4),
    dendrogram=False,
)

### Regulon AUC across CD4 subtypes — violin

In [ ]:
active_regs = [r for r in regulon_names
               if adata_cd4.obs[r].max() > 1e-5]

# Pick regulons with meaningful variation across CD4 subtypes
reg_var = adata_cd4.obs[active_regs].groupby(
    adata_cd4.obs['mq_l2']).mean().var()
top_regs_cd4 = reg_var.sort_values(ascending=False).head(12).index.tolist()

ncols = 4
nrows = int(np.ceil(len(top_regs_cd4) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.flatten()

for i, reg in enumerate(top_regs_cd4):
    df_r = adata_cd4.obs[['mq_l2','condition', reg]].copy()
    df_r['mq_l2'] = pd.Categorical(df_r['mq_l2'],
                                    categories=CD4_ORDER, ordered=True)
    sns.violinplot(
        data=df_r, x='mq_l2', y=reg, hue='condition',
        hue_order=['HD', 'PID7'], palette=PALETTE,
        split=True, inner='quartile', cut=0,
        order=CD4_ORDER, ax=axes[i], linewidth=0.8,
    )
    color = '#D65F5F' if reg in AP1_RELATED else 'black'
    axes[i].set_title(reg + (' ★' if reg in AP1_RELATED else ''),
                      fontsize=9, color=color)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('AUC', fontsize=8)
    axes[i].tick_params(axis='x', rotation=30, labelsize=7)
    axes[i].get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right',
           title='Condition', fontsize=8, title_fontsize=8)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Top variable regulons in CD4 T cells (★ = AP1/JunB)', y=1.01)
plt.tight_layout()
plt.show()

### AP1/JunB regulons — CD4 subsets, PID7 vs HD

In [ ]:
# Bar chart: mean AUC difference (PID7 − HD) per CD4 subtype per AP1 regulon
# IMPORTANT: Only plot cell types with ≥5 cells in BOTH conditions to avoid
# spurious differences driven by condition imbalance (e.g., CD4 Proliferating: 33 HD, 0 PID7)

min_cells_per_condition = 5
fig, axes = plt.subplots(1, len(AP1_RELATED),
                         figsize=(len(AP1_RELATED) * 4, 4))

for ax, reg in zip(axes, AP1_RELATED):
    df_reg_raw = (adata_cd4.obs[['mq_l2','condition', reg]]
                  .dropna(subset=['mq_l2'])
                  .groupby(['mq_l2','condition'])[reg]
                  .mean().unstack('condition', fill_value=np.nan)
                  .assign(diff=lambda d: d.get('PID7', np.nan) - d.get('HD', np.nan)))
    
    # Filter: keep only rows with both HD and PID7 means (no NaN)
    df_reg = df_reg_raw.dropna(subset=['HD','PID7','diff'])
    
    # Also filter: keep only cell types with ≥min_cells in both conditions
    ct_counts = adata_cd4.obs[['mq_l2','condition']].groupby(['mq_l2','condition']).size().unstack(fill_value=0)
    valid_cts = ct_counts[(ct_counts['HD'] >= min_cells_per_condition) & 
                          (ct_counts['PID7'] >= min_cells_per_condition)].index.tolist()
    df_reg = df_reg.loc[df_reg.index.isin(valid_cts)]
    
    # Reorder by CD4_ORDER (maintain biological order, not numerical)
    df_reg = df_reg.reindex([c for c in CD4_ORDER if c in df_reg.index])
    
    if len(df_reg) > 0:
        colors = ['#D65F5F' if v > 0 else '#4878CF' for v in df_reg['diff']]
        ax.barh(df_reg.index, df_reg['diff'], color=colors)
        ax.axvline(0, color='black', lw=0.8)
        ax.set_xlabel('Mean AUC (PID7 − HD)')
        ax.set_title(reg, color='#D65F5F', fontsize=9)
        ax.tick_params(axis='y', labelsize=8)
    else:
        ax.text(0.5, 0.5, 'No cell types with\n≥5 cells/condition',
                ha='center', va='center', transform=ax.transAxes, fontsize=8)
        ax.set_title(reg, color='#D65F5F', fontsize=9)

plt.suptitle(f'AP1/JunB regulons in CD4: PID7 − HD (min {min_cells_per_condition} cells/condition)', y=1.02)
plt.tight_layout()
plt.show()

# Scatter plot: HD vs PID7 regulon AUC for each CD4 subtype
# (Already filters out cell types with 0 PID7 cells via the sub logic below)
active_regs = [r for r in regulon_names if adata_cd4.obs[r].max() > 1e-5]
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, ct in zip(axes, CD4_ORDER):
    sub = adata_cd4.obs[adata_cd4.obs['mq_l2']==ct]
    n_hd  = (sub['condition']=='HD').sum()
    n_pid = (sub['condition']=='PID7').sum()
    
    if n_hd >= min_cells_per_condition and n_pid >= min_cells_per_condition:
        hd_m  = sub[sub['condition']=='HD'][active_regs].mean()
        pid_m = sub[sub['condition']=='PID7'][active_regs].mean()
        lim = max(hd_m.max(), pid_m.max()) * 1.1
        ax.scatter(hd_m, pid_m, s=30, alpha=0.7, color='#888888')
        ax.plot([0, lim], [0, lim], 'k--', lw=0.7)
        # Label AP1 regulons
        for reg in AP1_RELATED:
            if reg in hd_m.index:
                ax.annotate(reg.replace('(+)',''), (hd_m[reg], pid_m[reg]),
                            fontsize=6, color='#D65F5F',
                            xytext=(3, 3), textcoords='offset points')
        ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    else:
        ax.text(0.5, 0.5, f'Insufficient:\nHD={n_hd}, PID7={n_pid}', 
                ha='center', va='center', transform=ax.transAxes, fontsize=8)
    
    ax.set_xlabel('Mean AUC — HD', fontsize=8)
    ax.set_ylabel('Mean AUC — PID7', fontsize=8)
    ax.set_title(ct, fontsize=8)

plt.suptitle('Regulon AUC: HD vs PID7 per CD4 subtype (only ≥5 cells/condition)', y=1.02)
plt.tight_layout()
plt.show()